In [12]:
"""
Updated since 'August Update'. The main updates are:
- reducing the size of the neural network model
- also incorporating a Boosted Decision Tree
- plotting loss curves

> log-uniform sampling 
> 
"""

import numpy as np
import rebound
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
plt.ioff() # Turn off interactive mode
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.fft import fft
import pandas as pd

In [13]:
#-----------------------------------------------------------------------------
# Script Variables & Physics Constants
#-----------------------------------------------------------------------------
NUM_SIMULATIONS = 5000
MAX_TRANSITS = 100

SOLAR_TO_EARTH = 332946.0
# Assuming 2*pi in simulation = 365.25 days
SIM_TIME_TO_DAYS = 365.25 / (2 * np.pi)

In [14]:
#-----------------------------------------------------------------------------
# Utility Functions
#-----------------------------------------------------------------------------

def detect_transits(sim, integration_time, dt, max_transits, planet_index):
    """
    Detects the times when a specified planet transits the star (crosses the y-axis
    while in front of the star) within a REBOUND simulation.

    Inputs:
        :param sim: the simulation being run
        :param integration_time: the 'time' the sim is to run for
        :param dt: the integration time step
        :param max_transits: max number of transits we want to detect
        :param planet_index: index of the planet we want to detect transits of
    Outputs:
        array of transit times
    """
    transit_times = []
    star = sim.particles[0]             # star is always the first element in our sim (or should be!)
    planet = sim.particles[planet_index]# specify the index of the planet in our sim whose transits we want to detect
    prev_y = planet.y - star.y

    time = 0
    while time < integration_time and len(transit_times) < max_transits:
        sim.integrate(sim.t + dt)           # progress simulation
        time = sim.t                        # get time from sim
        curr_y = planet.y - star.y
        # if the product of previous and current y is negative, they they are on
        # opposite sides of the y axis, and so there has been a transit of the y-axis
        # if (planet.x - star.x) > 0, then the planet is in front of the star: transit occured
        if prev_y * curr_y < 0 and (planet.x - star.x) > 0:
            # perform linear interpolation to find more precise moment of crossing
            weight = -prev_y / (curr_y - prev_y)
            precise_time = (time - dt) + (weight * dt)
            transit_times.append(precise_time)
        prev_y = curr_y
    return np.array(transit_times)

def detrend_ttv(ttv_data):
    """
    Remove the linear trend from a single TTV array

    Inputs:
        array: ttv data
    Outputs:
        array: detrended ttv data
    """
    N = len(ttv_data)                                   # get length of input
    transit_numbers = np.arange(N)                      # create array of transit numbers
    P = np.polyfit(transit_numbers, ttv_data, 1)        # fit straight line to ttv data
    return ttv_data - (P[0] * transit_numbers + P[1])   # subtract line from ttv and return

def compute_oc_residuals(transit_times, max_transits):
    """
    Compute O-C residuals by fitting a linear ephemeris to observed transit times.

    Inputs:
        transit_times: Array of observed (perturbed) transit times
        max_transits: Maximum number of transits to use
    Outputs:
        Array of detrended O-C residuals
    """
    # Use only the first `max_transits` transits
    transit_times = transit_times[:max_transits]
    transit_numbers = np.arange(max_transits)  # [0, 1, 2, ..., max_transits-1]

    # Fit linear ephemeris: t = t0 + P * n
    # np.polyfit returns [slope (P), intercept (t0)] for degree=1
    P_est, t0_est = np.polyfit(transit_numbers, transit_times, 1)

    # Calculate the linear ephemeris
    linear_ephemeris = P_est * transit_numbers + t0_est

    # Compute O-C residuals: observed - calculated
    oc_residuals = transit_times - linear_ephemeris

    # Detrend the residuals (optional, but common in TTV analysis)
    return detrend_ttv(oc_residuals)

Change: Uniform Sampling on Log Masses. Ranges also changed.

In [15]:
#-----------------------------------------------------------------------------
# PHASE 1: GENERATE DATA WITH PERIODOGRAM
#-----------------------------------------------------------------------------

def generate_simulation_data(
    num_simulations,
    max_transits,
    resonances=None,
    resonance_fraction=1.0,
    add_noise=False,
    noise_level=0.01,
    missing_transits=False,
    missing_fraction=0.1
):
    """
    Generate simulation data with optional challenges (noise, missing transits, custom resonances).

    Inputs:
        num_simulations: Number of simulations to run.
        max_transits: Max number of transits to detect.
        resonances: List of resonance ratios (e.g., [1.5, 2.0, 3.0]). If None, use default.
        resonance_fraction: sets the fraction of resonant systems that will be generated
        add_noise: If True, add Gaussian noise to transit times.
        noise_level: Fraction of P_b to use as noise std (e.g., 0.01 = 1% of P_b).
        missing_transits: If True, randomly drop some transits.
        missing_fraction: Fraction of transits to drop (e.g., 0.1 = 10%).
    Outputs:
        all_features, all_masses, all_ttvs (as before)
    """
    print(f"Generating {num_simulations} simulations...")
    all_features = []
    all_masses = []
    all_ttvs = []

    # Default resonances if none provided
    if resonances is None:
        resonances = [1.5, 2.0, 3.0]

    for i in range(num_simulations):
        if (i + 1) % 50 == 0:
            print(f"  Simulation {i+1}...")

        try:
            # Randomize Planet B (The Transitor)
            # CHANGE: LOG UNIFORM FIX for proper mass sampling distribution
            m_b = 10 ** np.random.uniform(np.log10(3e-6), np.log10(9.55e-4))  # Mass (Terrestrial, Super-Earths to mini-Neptunes)
            P_b = np.random.uniform(2*np.pi, 4*np.pi)  # Period
            e_b = np.random.uniform(0.0, 0.08)  # Eccentricity

            # Randomize Planet C (The Perturber)
            if np.random.rand() < resonance_fraction:
                ratio = np.random.choice(resonances) + np.random.uniform(-0.02, 0.02)  # Tighter resonance
            else:
                ratio = np.random.uniform(1.1, 4.0)  # Wider range for non-resonant systems
            
            # LOG UNIFORM FIX for proper mass sampling distribution
            m_c = 10 ** np.random.uniform(np.log10(1e-4), np.log10(5e-3))  # Mass (1 Mjup to 5 Mjup)
            P_c = P_b * ratio  # Period
            e_c = np.random.uniform(0.0, 0.01)  # Eccentricity

            dt = min(P_b, P_c) / 100  # Use the smaller of P_b or P_c

            # Perturbed Simulation (no unperturbed simulation needed)
            sim_p = rebound.Simulation()
            sim_p.integrator = "whfast"
            sim_p.dt = dt
            sim_p.add(m=1.0)  # Star
            sim_p.add(m=m_b, P=P_b, e=e_b)  # Planet B
            sim_p.add(m=m_c, P=P_c, e=e_c)  # Planet C
            sim_p.move_to_com()
            transits_p = detect_transits(
                sim_p,
                P_b * (max_transits + 5),
                dt,
                max_transits,
                1
            )

            # Add noise to transit times if requested
            if add_noise:
                transits_p += np.random.normal(0, noise_level * P_b, size=len(transits_p))

            # Randomly drop transits if requested
            if missing_transits and len(transits_p) > max_transits:
                num_to_keep = int(len(transits_p) * (1 - missing_fraction))
                keep_indices = np.random.choice(len(transits_p), size=num_to_keep, replace=False)
                transits_p = transits_p[np.sort(keep_indices)]

            # Proceed if we have enough transits
            if len(transits_p) >= max_transits:
                # Compute O-C residuals (realistic TTVs)
                ttv_vec = compute_oc_residuals(transits_p, max_transits)

                # Feature construction
                #freqs = np.linspace(1/50, 1/3, 40)
                fft_coeffs = fft(ttv_vec)[:20]  # Take first 20 complex coefficients
                amp = np.std(ttv_vec)
                phys = [P_b, ratio, e_b]  # Note: m_b removed to avoid leakage
                features = np.hstack(([amp], np.real(fft_coeffs), np.imag(fft_coeffs), phys))

                all_features.append(features)
                all_masses.append(m_c)
                all_ttvs.append(ttv_vec)

        except Exception as e:
            print(f"Exception in simulation {i}: {e}. Proceeding to next simulation.")
            continue

    return np.array(all_features), np.array(all_masses), np.array(all_ttvs)

NN, Quantile

In [16]:
#-----------------------------------------------------------------------------
# PHASE 2: MASS PREDICTOR MODEL
#-----------------------------------------------------------------------------

class MassPredictor(nn.Module):
    """
    Neural Network model designed to estimate planetary mass from
    TTV amplitudes and periodogram features.
    Because we're working with normalised data and will have negative values,
    use LeakyReLU to stop neurons "dying".
    """
    def __init__(self, input_size):
        super(MassPredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            # nn.BatchNorm1d(32),        # Stabilizes training by normalizing layer outputs
            nn.LeakyReLU(0.1),          # Allows small gradients for negative values to prevent "dead neurons"
            nn.Dropout(0.1),            # Randomly zeros 10% of neurons to prevent overfitting to specific noise

            nn.Linear(32, 16),
            # nn.BatchNorm1d(16),
            nn.LeakyReLU(0.1),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

class QuantilePredictor(nn.Module):
    """
    Neural Network for predicting mass percentiles (uncertainty estimation).
    Outputs: [10th percentile, 50th percentile (median), 90th percentile].
    """
    def __init__(self, input_size):
        super(QuantilePredictor, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.LeakyReLU(0.1),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.1),
            nn.Linear(16, 3)  # Predict 3 quantiles (10th, 50th, 90th)
        )

    def forward(self, x):
        return self.network(x)

# Custom loss for quantile regression (pinball loss)
def quantile_loss(preds, targets, quantiles=[0.1, 0.5, 0.9]):
    loss = 0
    for i, q in enumerate(quantiles):
        errors = targets - preds[:, i]
        loss += torch.mean(torch.max(q * errors, (q - 1) * errors))
    return loss

# Training and Validation 

In [17]:
def train_model(model, X_train, y_train, X_val, y_val, epochs=200, batch_size=16):
    print("\nStarting model training...")
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

    criterion = nn.HuberLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

    best_val_loss, patience_counter, max_patience = float('inf'), 0, 30
    best_model_state = None
    train_loss_history, val_loss_history = [], []

    for epoch in range(epochs):
        model.train() 
        train_loss = 0
        indices = torch.randperm(len(X_train_t)) 

        for i in range(0, len(X_train_t), batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_X, batch_y = X_train_t[batch_indices], y_train_t[batch_indices]
            
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            
            optimizer.zero_grad()   
            loss.backward()         
            optimizer.step()        
            train_loss += loss.item()

        avg_train_loss = train_loss / len(X_train_t)

        model.eval() 
        with torch.no_grad():   
            val_loss = criterion(model(X_val_t), y_val_t)

        train_loss_history.append(avg_train_loss)
        val_loss_history.append(val_loss.item())

        scheduler.step(val_loss) 
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Val Loss: {val_loss.item():.6f}')

        if val_loss < best_val_loss:
            best_val_loss, patience_counter = val_loss, 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    if best_model_state: model.load_state_dict(best_model_state)
    return model, train_loss_history, val_loss_history

def train_quantile_model(model, X_train, y_train, X_val, y_val, epochs=200, batch_size=16):
    print("\nStarting quantile model training...")
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)  
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

    quantiles = [0.1, 0.5, 0.9]
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)

    best_val_loss, patience_counter, max_patience = float('inf'), 0, 30
    best_model_state = None
    train_loss_history, val_loss_history = [], []

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        indices = torch.randperm(len(X_train_t))

        for i in range(0, len(X_train_t), batch_size):
            batch_indices = indices[i:i+batch_size]
            batch_X, batch_y = X_train_t[batch_indices], y_train_t[batch_indices]
            outputs = model(batch_X)
            loss = quantile_loss(outputs, batch_y, quantiles)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        avg_train_loss = train_loss / len(X_train_t)

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)  
            val_loss = quantile_loss(val_outputs, y_val_t, quantiles)

        train_loss_history.append(avg_train_loss)
        val_loss_history.append(val_loss.item())

        scheduler.step(val_loss)
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Val Loss: {val_loss.item():.6f}')

        if val_loss < best_val_loss:
            best_val_loss, patience_counter = val_loss, 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                break

    if best_model_state: model.load_state_dict(best_model_state)
    return model, train_loss_history, val_loss_history

Changes Made: **Stratification** -- Since we are drawing data randomly from a wide distribution, a random 20% of it for a test set, test set might accidentally consist of mostly heavy planets and very few light ones. **Stratification** fixes this by organizing the dataset into "bins" (grouping by mass) before splitting. It guarantees that your Training, Validation, and Testing sets all contain the exact same distribution of planetary masses.

In [ ]:
#-----------------------------------------------------------------------------
# PHASE 3: TRAINING AND EVALUATION (REWRITTEN FOR READABILITY)
#-----------------------------------------------------------------------------

if __name__ == "__main__":
    
    np.random.seed(42)
    torch.manual_seed(42)
    
    print("\n--- Step 1: Generating All Simulations ---")
    X_all, y_all, ttvs_all = generate_simulation_data(
        num_simulations=NUM_SIMULATIONS,
        max_transits=MAX_TRANSITS,
        resonances=[1.5, 2.0, 3.0],
        resonance_fraction=0.5
    )

    print("\n--- Step 2: Creating a 20% Holdout Set from All Simulations (80% Pool) ---")
    # CHANGE: Stratify by mass for balanced datasets
    # np.digitize(input array, array of bins)
    # mass_bins_all = np.digitize(y_all, np.logspace(np.log10(min(y_all)), np.log10(max(y_all)), 10))
    # pd.qcut divides the data into 10 bins with an equal number of planets in each bin
    mass_bins_all = pd.qcut(y_all, q=10, labels=False)
    X_pool, X_holdout, y_pool, y_holdout = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42, stratify=mass_bins_all
    )

    print("\n--- Step 3: Four Independently-Generated Stress Tests ---")
    X_resonance,    y_resonance, _    = generate_simulation_data(200, MAX_TRANSITS, resonances=[4/3, 5/4, 3/2], resonance_fraction=1)
    X_nores, y_nores, _ = generate_simulation_data(200, MAX_TRANSITS, resonances=None, resonance_fraction=0)
    X_noisy, y_noisy, _  = generate_simulation_data(200, MAX_TRANSITS, add_noise=True, noise_level=0.01)
    X_missing, y_missing, _ = generate_simulation_data(200, MAX_TRANSITS, missing_transits=True, missing_fraction=0.1)

    print("\n--- Step 3.5: Log Transform All Target Variables ---")
    y_pool_log     = np.log10(y_pool)
    y_holdout_log = np.log10(y_holdout)
    y_resonance_log     = np.log10(y_resonance)
    y_nores_log  = np.log10(y_nores)
    y_noisy_log   = np.log10(y_noisy)
    y_missing_log = np.log10(y_missing)

    print("\n--- Step 4: Split Pool into Train & Val Sets ---")
    #mass_bins_trainval = np.digitize(y_pool, np.logspace(np.log10(min(y_pool)), np.log10(max(y_pool)), 10))
    # pd.qcut divides the data into 10 bins with an equal number of planets in each bin
    mass_bins_trainval = pd.qcut(y_pool, q=10, labels=False)
    X_train, X_val, y_train_log, y_val_log = train_test_split(
        X_pool, y_pool_log, test_size=0.2, random_state=42, stratify=mass_bins_trainval
    )

    print("\n--- Step 5: Scale Everything Based on Train Fit ---")
    scaler = StandardScaler().fit(X_train)

    X_train_s        = scaler.transform(X_train)
    X_val_s          = scaler.transform(X_val)
    X_holdout_s      = scaler.transform(X_holdout)
    X_resonance_s    = scaler.transform(X_resonance)
    X_nores_s        = scaler.transform(X_nores)
    X_noisy_s        = scaler.transform(X_noisy)
    X_missing_s      = scaler.transform(X_missing)


    # --- Train Models ---
    print("\n--- Training Standard Neural Network ---")
    model_point = MassPredictor(X_train_s.shape[1])
    model_nn, nn_train_loss, nn_val_loss = train_model(
        model_point, X_train_s, y_train_log, X_val_s, y_val_log, epochs=200
    )

    print("\n--- Training Quantile Neural Network ---")
    model_quantile = QuantilePredictor(X_train_s.shape[1])
    model_quantile, q_train_loss, q_val_loss = train_quantile_model(
        model_quantile, X_train_s, y_train_log, X_val_s, y_val_log, epochs=200
    )

    print("\n--- Training XGBoost Model ---")
    model_bdt = xgb.XGBRegressor(
        n_estimators=500, max_depth=6, learning_rate=0.05, 
        objective="reg:squarederror", eval_metric="rmse", random_state=42
    )
    # XGBoost trains fine on raw, unscaled tabular data, but we pass it anyway
    model_bdt.fit(
        X_train, y_train_log,
        eval_set=[(X_train, y_train_log), (X_val, y_val_log)],
        verbose=False 
    )

    # --- Evaluation Functions ---
    def evaluate_model(model, X_test, y_test, model_type="nn"):
        """Evaluate a model on a test set and return MAE."""
        if model_type == "nn":
            model.eval()
            with torch.no_grad():
                preds = model(torch.tensor(X_test, dtype=torch.float32)).numpy().flatten()
        else: 
            preds = model.predict(X_test)
        mae = mean_absolute_error(y_test, preds)
        return mae, preds

    def evaluate_quantile_model(model, X_test, y_test):
        """Evaluate BNN coverage and return MAE."""
        model.eval()
        with torch.no_grad():
            preds = model(torch.tensor(X_test, dtype=torch.float32)).numpy()  
        mae_median = mean_absolute_error(y_test, preds[:, 1])
        lower, upper = preds[:, 0], preds[:, 2] 
        coverage = np.mean((y_test >= lower) & (y_test <= upper)) * 100
        range_width = np.mean(upper - lower)
        return mae_median, coverage, range_width, preds


    # Evaluate Neural Network on all Stress Tests
    mae_holdout_nn, _ = evaluate_model(model_nn, X_holdout_s, y_holdout_log, "nn")
    mae_res_nn, _     = evaluate_model(model_nn, X_resonance_s,     y_resonance_log, "nn")
    mae_nonres_nn, _  = evaluate_model(model_nn, X_nores_s,  y_nores_log,  "nn")
    mae_noisy_nn, _   = evaluate_model(model_nn, X_noisy_s,   y_noisy_log,   "nn")
    mae_missing_nn, _ = evaluate_model(model_nn, X_missing_s, y_missing_log, "nn")

    # Evaluate Quantile Predictor
    mae_holdout_qp, cov_holdout_qp, rng_holdout_qp, _ = evaluate_quantile_model(model_quantile, X_holdout_s, y_holdout_log)
    mae_res_qp,     cov_res_qp,     rng_res_qp, _     = evaluate_quantile_model(model_quantile, X_resonance_s, y_resonance_log)
    mae_nonres_qp,  cov_nonres_qp,  rng_nonres_qp, _  = evaluate_quantile_model(model_quantile, X_nores_s,  y_nores_log)
    mae_noisy_qp,   cov_noisy_qp,   rng_noisy_qp, _   = evaluate_quantile_model(model_quantile, X_noisy_s,   y_noisy_log)
    mae_missing_qp, cov_missing_qp, rng_missing_qp, _ = evaluate_quantile_model(model_quantile, X_missing_s, y_missing_log)

    # Evaluate XGBoost (Testing on RAW unscaled data, because XGBoost works best unscaled)
    mae_holdout_bdt, _ = evaluate_model(model_bdt, X_holdout, y_holdout_log, "xgb")
    mae_res_bdt, _     = evaluate_model(model_bdt, X_resonance, y_resonance_log,     "xgb")
    mae_nonres_bdt, _  = evaluate_model(model_bdt, X_nores,  y_nores_log,  "xgb")
    mae_noisy_bdt, _   = evaluate_model(model_bdt, X_noisy,   y_noisy_log,   "xgb")
    mae_missing_bdt, _ = evaluate_model(model_bdt, X_missing, y_missing_log, "xgb")

    # --- Print Performance Summaries ---
    print("\n--- Neural Network Performance ---")
    print(f"{'Test Set':<25} {'MAE (dex)':<15}")
    print("-" * 40)
    print(f"{'Holdout Set':<25} {mae_holdout_nn:.4f}")
    print(f"{'Unseen Resonances':<25} {mae_res_nn:.4f}")
    print(f"{'Non-Resonant Systems':<25} {mae_nonres_nn:.4f}")
    print(f"{'Noisy Data':<25} {mae_noisy_nn:.4f}")
    print(f"{'Missing Transits':<25} {mae_missing_nn:.4f}")

    print("\n--- XGBoost Performance ---")
    print(f"{'Test Set':<25} {'MAE (dex)':<15}")
    print("-" * 40)
    print(f"{'Holdout Set':<25} {mae_holdout_bdt:.4f}")
    print(f"{'Unseen Resonances':<25} {mae_res_bdt:.4f}")
    print(f"{'Non-Resonant Systems':<25} {mae_nonres_bdt:.4f}")
    print(f"{'Noisy Data':<25} {mae_noisy_bdt:.4f}")
    print(f"{'Missing Transits':<25} {mae_missing_bdt:.4f}")

    print("\n--- Quantile Predictor Performance ---")
    print(f"{'Test Set':<25} {'MAE (dex)':<15} {'Coverage (%)':<15} {'Range Width (dex)':<20}")
    print("-" * 75)
    print(f"{'Holdout Set':<25} {mae_holdout_qp:<15.4f} {cov_holdout_qp:<15.1f} {rng_holdout_qp:<20.4f}")
    print(f"{'Unseen Resonances':<25} {mae_res_qp:<15.4f} {cov_res_qp:<15.1f} {rng_res_qp:<20.4f}")
    print(f"{'Non-Resonant Systems':<25} {mae_nonres_qp:<15.4f} {cov_nonres_qp:<15.1f} {rng_nonres_qp:<20.4f}")
    print(f"{'Noisy Data':<25} {mae_noisy_qp:<15.4f} {cov_noisy_qp:<15.1f} {rng_noisy_qp:<20.4f}")
    print(f"{'Missing Transits':<25} {mae_missing_qp:<15.4f} {cov_missing_qp:<15.1f} {rng_missing_qp:<20.4f}")

    # --- FINAL COMPARISON ---
    model_nn.eval()
    with torch.no_grad():
        preds_nn = model_nn(torch.tensor(X_holdout_s, dtype=torch.float32)).numpy().flatten()
    preds_bdt = model_bdt.predict(X_holdout) 
    
    mae_nn = mean_absolute_error(y_holdout_log, preds_nn)
    mae_bdt = mean_absolute_error(y_holdout_log, preds_bdt)

    print("\n" + "="*30)
    print(f"FINAL PERFORMANCE ({NUM_SIMULATIONS} SIMS, {MAX_TRANSITS} TRANSITS EACH)")
    print(f"Neural Network MAE: {mae_nn:.4f} dex")
    print(f"XGBoost BDT MAE:    {mae_bdt:.4f} dex")
    print("="*30)

    # -----------------------------------------------------------------------------
    # FINAL SIDE-BY-SIDE COMPARISON PLOTS
    # -----------------------------------------------------------------------------

    # Visualize sample TTV signal
    plt.figure(figsize=(10, 6))
    plt.plot(ttvs_all[0] * SIM_TIME_TO_DAYS, 'o-', label='TTV Signal')
    plt.title(f'Sample TTV Signal (Mass = {y_pool[0] * SOLAR_TO_EARTH:.2f} Earth Masses)')
    plt.xlabel('Transit Number')
    plt.ylabel('Time Variation (Days)')
    plt.legend()
    plt.grid(True)

    # --- Feature Importance Analysis ---
    feature_names = (
            ["TTV Amp"] +
            [f"FFT_Real_{i}" for i in range(20)] +  
            [f"FFT_Imag_{i}" for i in range(20)] +  
            ["Period_B", "Period_Ratio", "Ecc_B"]
    )
    importances = model_bdt.feature_importances_
    indices = np.argsort(importances)[-10:] 

    plt.figure(figsize=(10, 6))
    plt.barh(range(len(indices)), importances[indices], align='center', color='teal')
    plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
    plt.xlabel("XGBoost Feature Importance Score")
    plt.title("Which features helped predict the Perturber Mass?")
    plt.tight_layout()


    # 1. LEARNING CURVE COMPARISON
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].plot(nn_train_loss, label='Train Loss', color='blue')
    ax[0].plot(nn_val_loss, label='Val Loss', color='red', linestyle='--')
    ax[0].set_title("Neural Network: Huber Loss")
    ax[0].set_xlabel("Epoch"); ax[0].legend()

    results = model_bdt.evals_result()
    ax[1].plot(results['validation_0']['rmse'], label='Train RMSE', color='green')
    ax[1].plot(results['validation_1']['rmse'], label='Val RMSE', color='orange', linestyle='--')
    ax[1].set_title("BDT: RMSE Learning Curve")
    ax[1].set_xlabel("Number of Trees"); ax[1].legend()

    plt.suptitle("Figure 1: Training Convergence & Overfitting Check", fontsize=14)
    plt.tight_layout()


    # 2. PREDICTED VS ACTUAL (The "Scatter" Test)
    y_holdout_earth = 10**y_holdout_log * SOLAR_TO_EARTH
    fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True, sharex=True)
    
    # NN
    ax[0].scatter(y_holdout_earth, 10**preds_nn * SOLAR_TO_EARTH, alpha=0.3, color='blue')
    ax[0].plot([y_holdout_earth.min(), y_holdout_earth.max()], [y_holdout_earth.min(), y_holdout_earth.max()], 'r--')
    ax[0].set_title(f"NN (MAE: {mae_nn:.3f} dex)")
    ax[0].set_xscale('log'); ax[0].set_yscale('log')
    ax[0].set_ylabel("Predicted Mass (Earths)")

    # BDT
    ax[1].scatter(y_holdout_earth, 10**preds_bdt * SOLAR_TO_EARTH, alpha=0.3, color='green')
    ax[1].plot([y_holdout_earth.min(), y_holdout_earth.max()], [y_holdout_earth.min(), y_holdout_earth.max()], 'r--')
    ax[1].set_title(f"BDT (MAE: {mae_bdt:.3f} dex)")
    
    plt.suptitle("Figure 2: Mass Prediction Accuracy (Holdout Set)", fontsize=14)
    plt.tight_layout()

    # 3. PERCENTAGE ERROR VS MASS
    fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    res_nn = (10 ** (preds_nn - y_holdout_log) - 1) * 100
    res_bdt = (10 ** (preds_bdt - y_holdout_log) - 1) * 100

    ax[0].scatter(y_holdout_earth, res_nn, alpha=0.4, color='purple')
    ax[0].axhline(0, color='black', ls='--')
    ax[0].set_title("NN Percentage Error")
    ax[0].set_xscale('log')

    ax[1].scatter(y_holdout_earth, res_bdt, alpha=0.4, color='orange')
    ax[1].axhline(0, color='black', ls='--')
    ax[1].set_title("BDT Percentage Error")
    ax[1].set_xscale('log')
    plt.suptitle("Figure 3: Error Trends across Mass Ranges", fontsize=14)


    # 4. ERROR DISTRIBUTION (Histograms)
    fig, ax = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
    ax[0].hist(res_nn, bins=30, color='blue', alpha=0.6, edgecolor='black')
    ax[0].set_title("NN Error Distribution")
    ax[0].set_xlabel("Error %")

    ax[1].hist(res_bdt, bins=30, color='green', alpha=0.6, edgecolor='black')
    ax[1].set_title("BDT Error Distribution")
    ax[1].set_xlabel("Error %")
    plt.suptitle("Figure 4: Bias and Variance Comparison", fontsize=14)


    # 5. RESONANCE ANALYSIS (Error vs Period Ratio)
    test_ratios = X_holdout[:, 42]
    resonances = [1.5, 2.0, 3.0]
    res_labels = ["3:2", "2:1", "3:1"]

    for i, axes in enumerate(ax):
        current_res = res_nn if i == 0 else res_bdt
        title = "NN: Error vs Resonance" if i == 0 else "BDT: Error vs Resonance"

        im = axes.scatter(test_ratios, np.abs(current_res), c=y_holdout_earth,
                          cmap='viridis', norm=mcolors.LogNorm(), alpha=0.5)

        for val, label in zip(resonances, res_labels):
            axes.axvline(val, color='red', linestyle='--', alpha=0.6, lw=1)
            axes.text(val, axes.get_ylim()[1]*0.9, label, color='red',
                      fontsize=9, ha='center', fontweight='bold')

        axes.set_title(title)
        axes.set_xlabel("Period Ratio ($P_c/P_b$)")
        if i == 0:
            axes.set_ylabel("Absolute Error (%)")

    plt.tight_layout(rect=[0, 0, 0.9, 1]) 
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='True Mass ($M_{Earth}$)')

    plt.suptitle("Figure 5: Physics Check - Model Performance near Orbital Resonances", fontsize=14, y=1.05)

    # Plot quantile predictions vs. actual
    preds_quantile = model_quantile(torch.tensor(X_holdout_s, dtype=torch.float32)).detach().numpy()
    lower_earth = 10 ** preds_quantile[:, 0] * SOLAR_TO_EARTH  
    median_earth = 10 ** preds_quantile[:, 1] * SOLAR_TO_EARTH  
    upper_earth = 10 ** preds_quantile[:, 2] * SOLAR_TO_EARTH  

    plt.figure(figsize=(10, 6))
    plt.scatter(y_holdout_earth, median_earth, alpha=0.3, color='blue', label='Median Prediction')
    plt.fill_between(
        y_holdout_earth,
        lower_earth,
        upper_earth,
        alpha=0.2,
        color='blue',
        label='10th–90th Percentile Range'
    )
    plt.plot([y_holdout_earth.min(), y_holdout_earth.max()], [y_holdout_earth.min(), y_holdout_earth.max()], 'r--')
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("Actual Mass (Earth Masses)")
    plt.ylabel("Predicted Mass (Earth Masses)")
    plt.title(f"Quantile Predictor: Median ± Uncertainty (Coverage: {cov_holdout_qp:.1f}%)")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.show()


--- Step 1: Generating All Simulations ---
Generating 5000 simulations...
  Simulation 50...
  Simulation 100...
  Simulation 150...
  Simulation 200...
  Simulation 250...
  Simulation 300...
  Simulation 350...
  Simulation 400...
  Simulation 450...
  Simulation 500...
  Simulation 550...
  Simulation 600...
  Simulation 650...
  Simulation 700...
  Simulation 750...
  Simulation 800...
  Simulation 850...
  Simulation 900...
  Simulation 950...
  Simulation 1000...
  Simulation 1050...
  Simulation 1100...
  Simulation 1150...
  Simulation 1200...
  Simulation 1250...
  Simulation 1300...
  Simulation 1350...
  Simulation 1400...
  Simulation 1450...
  Simulation 1500...
  Simulation 1550...
  Simulation 1600...
  Simulation 1650...
  Simulation 1700...
  Simulation 1750...
  Simulation 1800...
  Simulation 1850...
  Simulation 1900...
  Simulation 1950...
  Simulation 2000...
  Simulation 2050...
  Simulation 2100...
  Simulation 2150...
  Simulation 2200...
  Simulation 2250...


d:\miniconda3\Lib\site-packages\rebound\simulation.py:264: RuntimeWarning: Possible convergence issue. Timestep in Kepler solver is larger than one orbital period.
  warnings.warn(msg[1:], RuntimeWarning)


  Simulation 2600...
  Simulation 2650...
  Simulation 2700...
  Simulation 2750...
  Simulation 2800...
  Simulation 2850...
  Simulation 2900...
  Simulation 2950...
  Simulation 3000...
  Simulation 3050...
  Simulation 3100...
  Simulation 3150...
  Simulation 3200...
  Simulation 3250...
  Simulation 3300...
  Simulation 3350...
  Simulation 3400...
  Simulation 3450...
  Simulation 3500...
  Simulation 3550...
  Simulation 3600...
  Simulation 3650...
  Simulation 3700...
  Simulation 3750...
  Simulation 3800...
  Simulation 3850...
  Simulation 3900...
  Simulation 3950...
  Simulation 4000...
  Simulation 4050...
  Simulation 4100...
  Simulation 4150...
  Simulation 4200...
  Simulation 4250...
  Simulation 4300...
  Simulation 4350...
  Simulation 4400...
  Simulation 4450...
  Simulation 4500...
  Simulation 4550...
  Simulation 4600...
  Simulation 4650...
  Simulation 4700...
  Simulation 4750...
  Simulation 4800...
  Simulation 4850...
  Simulation 4900...
  Simulation 